# Feed forward with GeLU activation

Advantages of GeLU over ReLU.
1. Differentiable. Each value of x corresponds to its different value.(not the same 0 for negatives)
2. Solves dead neurons problem, as negative values will also contribute to training as it is not 0.
3. As per experiments, GeLU provided great results compared to other activation functions in GPTs.

In [11]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2 / torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
    ))

In [12]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # vocabulary size
    "context_length": 1024, # context length
    "emb_dim": 768, # embedding dimension
    "n_heads": 12,  # no. of attention heads
    "n_layers": 12, # no. of transformer layers
    "drop_rate": 0.1, # dropout rate
    "qkv_bias": False # query-key-value bias
}

In [13]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), # Expansion
        GELU(), # Activation
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), # Contraction
    )

  def forward(self, x):
    return self.layers(x)

In [14]:
print(GPT_CONFIG_124M["emb_dim"])

768


In [16]:
ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2,3,768)
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])
